# 💄 Sephora Ürün Öneri Sistemi
> NLP Tabanlı Sentiment Analizi & Kişiselleştirilmiş Ürün Öneri Pipeline

---
**Pipeline Özeti:**
1. Veri yükleme & keşifsel analiz (EDA)
2. Feature engineering (metin + sayısal + ürün bazlı)
3. Feature seçimi (SelectKBest, RFE, Feature Importance)
4. Çoklu model eğitimi + hyperparameter search
5. Metrik karşılaştırma & en iyi modeli kaydetme
6. Ürün öneri motoru (müşteri cilt tipi + şikayeti → Top-5)
7. Sonuçları `outputs/models/` ve `outputs/metrics/` altına kaydetme

## 0. Kütüphaneler & Konfigürasyon

In [ ]:
# ── Standart & Bilimsel ──────────────────────────────────────────────────────
import json
import os
import pickle
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# ── Görselleştirme ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

# ── Sklearn ──────────────────────────────────────────────────────────────────
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import RFE, SelectKBest, chi2, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.svm import LinearSVC

# ── Cosine Similarity (öneri motoru) ─────────────────────────────────────────
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')

# ── Çıktı dizinleri ──────────────────────────────────────────────────────────
OUTPUT_ROOT = Path('outputs')
MODELS_DIR  = OUTPUT_ROOT / 'models'
METRICS_DIR = OUTPUT_ROOT / 'metrics'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

# ── Plot stili ───────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5)})

RANDOM_STATE = 42
print('✅ Kütüphaneler yüklendi.')
print(f'   Modeller  → {MODELS_DIR}')
print(f'   Metrikler → {METRICS_DIR}')

## 1. Veri Yükleme

In [ ]:
# ─── Veri setinizin yolunu buraya girin ──────────────────────────────────────
DATA_PATH = 'sephora_reviews.csv'   # ← kendi dosya adınızla değiştirin

df_raw = pd.read_csv(DATA_PATH)
print(f'Veri boyutu : {df_raw.shape}')
print(f'Sütunlar    : {list(df_raw.columns)}')
df_raw.head(3)

## 2. Keşifsel Veri Analizi (EDA)

In [ ]:
# ── Genel istatistikler ───────────────────────────────────────────────────────
print('=== Eksik Değerler ===')
missing = df_raw.isnull().sum()
display(missing[missing > 0].sort_values(ascending=False).to_frame('eksik'))

print('\n=== Hedef Değişken Dağılımı ===')
print(df_raw['rating_category'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Rating dağılımı
df_raw['rating'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Rating Dağılımı'); axes[0].set_xlabel('Rating')

# Kategori dağılımı
df_raw['primary_category'].value_counts().head(10).plot(
    kind='barh', ax=axes[1], color='salmon', edgecolor='white')
axes[1].set_title('Top-10 Kategori'); axes[1].invert_yaxis()

# Fiyat dağılımı
df_raw['price_usd'].dropna().plot(
    kind='hist', bins=40, ax=axes[2], color='mediumpurple', edgecolor='white')
axes[2].set_title('Fiyat Dağılımı ($)')

plt.tight_layout()
plt.savefig(METRICS_DIR / 'eda_distributions.png', bbox_inches='tight')
plt.show()
print('📊 EDA grafikleri kaydedildi.')

## 3. Veri Hazırlama & NLP Ön İşleme

In [ ]:
# ── Mevcut NLP modülünüzdeki prepare fonksiyonu ───────────────────────────────
def prepare_sentiment_dataset(df: pd.DataFrame) -> pd.DataFrame:
    required_cols = [
        'product_id', 'brand_name', 'product_name',
        'primary_category', 'secondary_category',
        'clean_text', 'rating', 'price_usd', 'rating_category'
    ]
    df_model = df[required_cols].copy()
    df_model = df_model[df_model['rating_category'].isin(['positive', 'negative'])].copy()
    df_model = df_model.dropna(subset=['clean_text'])
    df_model['clean_text'] = df_model['clean_text'].astype(str)
    return df_model

df_model = prepare_sentiment_dataset(df_raw)
print(f'Model veri seti: {df_model.shape}')
print(df_model['rating_category'].value_counts())
df_model.head(3)

## 4. Feature Engineering

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4.1  Metin Özellikleri
# ══════════════════════════════════════════════════════════════════════════════

def add_text_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Yorum metninden istatistiksel ve kural tabanlı özellikler türetir.
    """
    df = df.copy()
    text = df['clean_text'].fillna('')

    df['text_length']       = text.str.len()
    df['word_count']        = text.str.split().str.len().fillna(0).astype(int)
    df['avg_word_length']   = text.apply(
        lambda t: np.mean([len(w) for w in t.split()]) if t.split() else 0)
    df['exclamation_count'] = text.str.count(r'!')
    df['question_count']    = text.str.count(r'\?')
    df['uppercase_ratio']   = text.apply(
        lambda t: sum(c.isupper() for c in t) / max(sum(c.isalpha() for c in t), 1))
    df['unique_word_ratio'] = text.apply(
        lambda t: len(set(t.lower().split())) / max(len(t.split()), 1))

    skin_pattern    = r'\b(oily|dry|combination|sensitive|normal|acne[\- ]?prone|mature)\b'
    concern_pattern = r'\b(acne|wrinkle|pore|dark.spot|hyperpigmentation|redness|dull|aging|moisture|hydrat|brightening)\b'
    ingredient_pattern = r'\b(retinol|niacinamide|hyaluronic|vitamin.c|aha|bha|salicylic|glycolic|ceramide|peptide)\b'

    df['skin_type_mention']  = text.str.contains(skin_pattern,      case=False, regex=True).astype(int)
    df['concern_mention']    = text.str.contains(concern_pattern,   case=False, regex=True).astype(int)
    df['ingredient_mention'] = text.str.contains(ingredient_pattern, case=False, regex=True).astype(int)

    return df


# ══════════════════════════════════════════════════════════════════════════════
# 4.2  Ürün / Sayısal Özellikler
# ══════════════════════════════════════════════════════════════════════════════

def add_product_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ürün bazlı aggregate ve fiyat özellikleri ekler.
    """
    df = df.copy()

    df['price_log'] = np.log1p(df['price_usd'].fillna(0))

    scaler = MinMaxScaler()
    df['price_scaled'] = df.groupby('primary_category')['price_usd'].transform(
        lambda x: scaler.fit_transform(x.fillna(x.median()).values.reshape(-1, 1)).flatten()
    )

    product_stats = (
        df.groupby('product_id')
        .agg(
            product_mean_rating = ('rating', 'mean'),
            review_count        = ('rating', 'count'),
            positive_ratio      = ('rating_category',
                                   lambda x: (x == 'positive').sum() / max(len(x), 1)),
            rating_std          = ('rating', 'std'),
        )
        .reset_index()
    )
    product_stats['rating_std'] = product_stats['rating_std'].fillna(0)

    df = df.merge(product_stats, on='product_id', how='left')
    df['rating_deviation'] = df['rating'] - df['product_mean_rating']

    return df


# ── Uygula ────────────────────────────────────────────────────────────────────
df_feat = add_text_features(df_model)
df_feat = add_product_features(df_feat)

NUMERIC_FEATURES = [
    'text_length', 'word_count', 'avg_word_length',
    'exclamation_count', 'question_count', 'uppercase_ratio', 'unique_word_ratio',
    'skin_type_mention', 'concern_mention', 'ingredient_mention',
    'price_log', 'price_scaled',
    'rating_deviation', 'review_count', 'positive_ratio',
    'product_mean_rating', 'rating_std',
]
NUMERIC_FEATURES = [c for c in NUMERIC_FEATURES if c in df_feat.columns]

print(f'Oluşturulan özellik sayısı : {len(NUMERIC_FEATURES)}')
df_feat[NUMERIC_FEATURES].describe().round(3)

## 5. Feature Seçimi

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 5.1  SelectKBest — Mutual Information
# ══════════════════════════════════════════════════════════════════════════════

le = LabelEncoder()
y_encoded = le.fit_transform(df_feat['rating_category'])

X_num = df_feat[NUMERIC_FEATURES].fillna(0)

selector = SelectKBest(score_func=mutual_info_classif, k='all')
selector.fit(X_num, y_encoded)

mi_scores = pd.Series(selector.scores_, index=NUMERIC_FEATURES).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
mi_scores.plot(kind='bar', color='teal', edgecolor='white')
plt.title('Mutual Information Skorları (Numeric Features)')
plt.ylabel('MI Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(METRICS_DIR / 'feature_importance_mi.png', bbox_inches='tight')
plt.show()

# Eşik üstü özellikleri seç
MI_THRESHOLD   = 0.01
SELECTED_FEATS = mi_scores[mi_scores >= MI_THRESHOLD].index.tolist()
print(f'Seçilen özellikler ({len(SELECTED_FEATS)}): {SELECTED_FEATS}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 5.2  RFE ile Özellik Önemi (LogReg tabanlı)
# ══════════════════════════════════════════════════════════════════════════════

rfe_estimator = LogisticRegression(max_iter=500, class_weight='balanced')
rfe = RFE(estimator=rfe_estimator, n_features_to_select=min(8, len(NUMERIC_FEATURES)), step=1)
rfe.fit(X_num, y_encoded)

rfe_selected = [f for f, s in zip(NUMERIC_FEATURES, rfe.support_) if s]
print('RFE ile seçilen özellikler:')
for f in rfe_selected:
    print(f'  ✔ {f}')

## 6. Train / Test Split

In [ ]:
X_text = df_feat['clean_text']
y      = df_feat['rating_category']

X_tr_txt, X_te_txt, y_train, y_test = train_test_split(
    X_text, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print(f'Train: {len(X_tr_txt):>6}  |  Test: {len(X_te_txt):>6}')
print('Train sınıf dağılımı:')
print(y_train.value_counts())

## 7. Model Eğitimi — Çoklu Model + Hyperparameter Search

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 7.1  Deney Kaydı
# ══════════════════════════════════════════════════════════════════════════════

def _tfidf_base(max_features=10_000, ngram_range=(1, 2)):
    return TfidfVectorizer(
        stop_words='english',
        max_features=max_features,
        ngram_range=ngram_range,
        sublinear_tf=True,
    )


EXPERIMENTS = {

    'logistic_regression': {
        'pipeline': Pipeline([
            ('tfidf',  _tfidf_base()),
            ('model',  LogisticRegression(max_iter=1000, class_weight='balanced')),
        ]),
        'param_grid': {
            'tfidf__max_features': [5_000, 10_000],
            'tfidf__ngram_range':  [(1, 1), (1, 2)],
            'model__C':            [0.1, 1.0, 10.0],
        },
    },

    'linear_svc': {
        'pipeline': Pipeline([
            ('tfidf',  _tfidf_base()),
            ('model',  LinearSVC(class_weight='balanced', max_iter=2000)),
        ]),
        'param_grid': {
            'tfidf__max_features': [5_000, 10_000],
            'tfidf__ngram_range':  [(1, 1), (1, 2)],
            'model__C':            [0.01, 0.1, 1.0],
        },
    },

    'random_forest': {
        'pipeline': Pipeline([
            ('tfidf',  _tfidf_base(max_features=5_000)),
            ('model',  RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE)),
        ]),
        'param_grid': {
            'tfidf__ngram_range':       [(1, 1), (1, 2)],
            'model__n_estimators':      [100, 200],
            'model__max_depth':         [None, 20],
            'model__min_samples_split': [2, 5],
        },
    },

    'naive_bayes': {
        'pipeline': Pipeline([
            ('tfidf',  _tfidf_base()),
            ('model',  MultinomialNB()),
        ]),
        'param_grid': {
            'tfidf__max_features': [5_000, 10_000],
            'tfidf__ngram_range':  [(1, 1), (1, 2)],
            'model__alpha':        [0.1, 0.5, 1.0],
        },
    },
}

print(f'{len(EXPERIMENTS)} model denenecek:', list(EXPERIMENTS.keys()))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 7.2  GridSearchCV Döngüsü
# ══════════════════════════════════════════════════════════════════════════════

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
all_results = {}

for model_name, exp in EXPERIMENTS.items():
    print(f'\n🔍  [{model_name}] GridSearchCV başlıyor...')
    t0 = time.time()

    grid = GridSearchCV(
        estimator  = exp['pipeline'],
        param_grid = exp['param_grid'],
        cv         = cv,
        scoring    = 'f1_weighted',
        n_jobs     = -1,
        verbose    = 0,
        refit      = True,
    )
    grid.fit(X_tr_txt, y_train)

    # ── Test seti değerlendirmesi ─────────────────────────────────────────────
    y_pred = grid.predict(X_te_txt)

    # ROC-AUC için predict_proba gerekir (SVC hariç)
    try:
        y_proba = grid.predict_proba(X_te_txt)[:, 1]
        auc = roc_auc_score((y_test == 'positive').astype(int), y_proba)
    except AttributeError:
        auc = None

    metrics = {
        'model':           model_name,
        'best_params':     grid.best_params_,
        'cv_f1':           round(grid.best_score_, 4),
        'test_accuracy':   round(accuracy_score(y_test, y_pred),               4),
        'test_f1':         round(f1_score(y_test, y_pred, average='weighted'), 4),
        'test_precision':  round(precision_score(y_test, y_pred, average='weighted', zero_division=0), 4),
        'test_recall':     round(recall_score(y_test, y_pred, average='weighted'),    4),
        'test_auc':        round(auc, 4) if auc is not None else 'N/A',
        'train_time_sec':  round(time.time() - t0, 1),
    }

    all_results[model_name] = {
        'metrics': metrics,
        'grid':    grid,
        'y_pred':  y_pred,
    }

    print(f'   ✅  CV F1={metrics["cv_f1"]}  |  Test F1={metrics["test_f1"]}  |  '
          f'Accuracy={metrics["test_accuracy"]}  |  Süre={metrics["train_time_sec"]}s')
    print(f'   En iyi params: {grid.best_params_}')

    # ── Metrikleri JSON olarak kaydet ─────────────────────────────────────────
    metrics_path = METRICS_DIR / f'{model_name}_metrics.json'
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=2, default=str)

print('\n🎉 Tüm modeller eğitildi!')

## 8. Model Karşılaştırması

In [ ]:
# ── Karşılaştırma tablosu ─────────────────────────────────────────────────────
rows = [r['metrics'] for r in all_results.values()]
comparison_df = pd.DataFrame(rows).set_index('model').drop(columns=['best_params'])

# En iyi modeli vurgula
best_model_name = comparison_df['test_f1'].idxmax()
print(f'🏆 En iyi model: {best_model_name}')
display(comparison_df.style
    .background_gradient(cmap='YlGn', subset=['test_f1', 'test_accuracy'])
    .highlight_max(subset=['test_f1'], color='#d4edda')
    .format(precision=4)
)

# Tüm karşılaştırma tablosunu kaydet
comparison_df.to_csv(METRICS_DIR / 'all_models_comparison.csv')
print(f'\n💾 Karşılaştırma tablosu kaydedildi → {METRICS_DIR / "all_models_comparison.csv"}')

In [ ]:
# ── Karşılaştırma grafiği ─────────────────────────────────────────────────────
metrics_to_plot = ['test_accuracy', 'test_f1', 'test_precision', 'test_recall']
plot_df = comparison_df[metrics_to_plot].copy()

ax = plot_df.plot(kind='bar', figsize=(12, 5), width=0.7, edgecolor='white')
ax.set_title('Model Karşılaştırması — Test Seti Metrikleri', fontsize=13)
ax.set_xticklabels(plot_df.index, rotation=20, ha='right')
ax.set_ylim(0.5, 1.0)
ax.legend(loc='lower right')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.savefig(METRICS_DIR / 'model_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Confusion Matrix — En İyi Model ──────────────────────────────────────────
best_result = all_results[best_model_name]
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, best_result['y_pred'],
    display_labels=['negative', 'positive'],
    cmap='Blues', ax=ax
)
ax.set_title(f'Confusion Matrix — {best_model_name}')
plt.tight_layout()
plt.savefig(METRICS_DIR / f'{best_model_name}_confusion_matrix.png', bbox_inches='tight')
plt.show()

# Classification report
print(f'\n{best_model_name} — Detaylı Sınıflandırma Raporu:')
print(classification_report(y_test, best_result['y_pred']))

In [ ]:
# ── ROC Eğrisi (prob destekleyen modeller) ────────────────────────────────────
plt.figure(figsize=(7, 5))
for model_name, result in all_results.items():
    try:
        proba = result['grid'].predict_proba(X_te_txt)[:, 1]
        fpr, tpr, _ = roc_curve((y_test == 'positive').astype(int), proba)
        auc_val = result['metrics']['test_auc']
        plt.plot(fpr, tpr, label=f'{model_name} (AUC={auc_val})')
    except AttributeError:
        pass

plt.plot([0, 1], [0, 1], 'k--', alpha=0.4)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Eğrileri')
plt.legend()
plt.tight_layout()
plt.savefig(METRICS_DIR / 'roc_curves.png', bbox_inches='tight')
plt.show()

## 9. En İyi Modeli Kaydet

In [ ]:
best_pipeline = best_result['grid'].best_estimator_

# ── Modeli pickle olarak kaydet ───────────────────────────────────────────────
model_path = MODELS_DIR / f'{best_model_name}_best.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(best_pipeline, f)

# ── Özet metrik dosyası ───────────────────────────────────────────────────────
summary = {
    'best_model':       best_model_name,
    'model_path':       str(model_path),
    'best_params':      all_results[best_model_name]['metrics']['best_params'],
    'test_f1':          all_results[best_model_name]['metrics']['test_f1'],
    'test_accuracy':    all_results[best_model_name]['metrics']['test_accuracy'],
    'all_model_scores': {k: v['metrics']['test_f1'] for k, v in all_results.items()},
}
with open(METRICS_DIR / 'best_model_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f'✅ En iyi model kaydedildi  → {model_path}')
print(f'✅ Özet metrikleri kaydedildi → {METRICS_DIR / "best_model_summary.json"}')
print(f'\n   Model : {best_model_name}')
print(f'   Test F1: {summary["test_f1"]}')
print(f'   Test Accuracy: {summary["test_accuracy"]}')

## 10. Ürün Öneri Motoru

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 10.1  Ürün profili oluştur
# ══════════════════════════════════════════════════════════════════════════════

# Her ürün için ağırlıklı sentiment skoru hesapla
df_feat['sentiment_score'] = best_pipeline.predict_proba(df_feat['clean_text'])[:, 1] \
    if hasattr(best_pipeline, 'predict_proba') \
    else (df_feat['rating_category'] == 'positive').astype(float)

df_feat['sentiment_pred'] = best_pipeline.predict(df_feat['clean_text'])

# Ürün bazlı özellikler
product_profiles = (
    df_feat.groupby('product_id')
    .agg(
        product_name         = ('product_name',     'first'),
        brand_name           = ('brand_name',        'first'),
        primary_category     = ('primary_category',  'first'),
        secondary_category   = ('secondary_category','first'),
        price_usd            = ('price_usd',         'median'),
        avg_sentiment_score  = ('sentiment_score',   'mean'),
        review_count         = ('sentiment_score',   'count'),
        avg_rating           = ('rating',            'mean'),
        positive_ratio       = ('rating_category',
                                lambda x: (x == 'positive').sum() / max(len(x), 1)),
        # Tüm yorumları birleştir (arama için)
        all_reviews_text     = ('clean_text',        lambda x: ' '.join(x.dropna().astype(str))),
    )
    .reset_index()
)

# Yeterli yorum olan ürünleri filtrele
MIN_REVIEWS = 3
product_profiles = product_profiles[product_profiles['review_count'] >= MIN_REVIEWS].copy()

print(f'Öneri havuzundaki ürün sayısı: {len(product_profiles)}')
product_profiles.head(3)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 10.2  TF-IDF Vektörleri — Yorum Tabanlı Benzerlik
# ══════════════════════════════════════════════════════════════════════════════

SKIN_TYPE_KEYWORDS = {
    'oily':        ['oily', 'shiny', 'greasy', 'pore', 'sebum'],
    'dry':         ['dry', 'flaky', 'tight', 'rough', 'dehydrated'],
    'combination': ['combination', 't-zone', 'mixed'],
    'sensitive':   ['sensitive', 'reactive', 'redness', 'irritat'],
    'normal':      ['normal', 'balanced'],
    'acne_prone':  ['acne', 'breakout', 'pimple', 'blemish', 'zit'],
}

CONCERN_KEYWORDS = {
    'anti_aging':      ['wrinkle', 'aging', 'fine line', 'firmness', 'mature'],
    'brightening':     ['dull', 'bright', 'glow', 'dark spot', 'hyperpigment', 'uneven'],
    'hydration':       ['moisture', 'hydrat', 'dry', 'plump', 'dewy'],
    'pore_minimizing': ['pore', 'minimize', 'refin', 'tight'],
    'acne_treatment':  ['acne', 'breakout', 'blemish', 'clarify', 'salicylic'],
    'soothing':        ['redness', 'calm', 'soothe', 'irritat', 'sensitive'],
}


def extract_skin_profile(user_input: str) -> dict:
    """Kullanıcı metnini cilt tipi + şikayet profiline dönüştürür."""
    text = user_input.lower()
    skin_types = [
        st for st, kws in SKIN_TYPE_KEYWORDS.items()
        if any(kw in text for kw in kws)
    ]
    concerns = [
        c for c, kws in CONCERN_KEYWORDS.items()
        if any(kw in text for kw in kws)
    ]
    return {
        'detected_skin_types': skin_types or ['unknown'],
        'detected_concerns':   concerns   or ['general'],
        'clean_input':         re.sub(r'[^a-z0-9 ]', ' ', text).strip(),
    }


# ── Ürün yorum korpusunu vektörleştir ────────────────────────────────────────
product_tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=15_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
)
product_vectors = product_tfidf.fit_transform(product_profiles['all_reviews_text'])
print(f'Ürün vektör matrisi: {product_vectors.shape}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 10.3  Öneri Fonksiyonu
# ══════════════════════════════════════════════════════════════════════════════

def recommend_products(
    user_input: str,
    top_n: int = 5,
    min_positive_ratio: float = 0.6,
    category_filter: str = None,
) -> pd.DataFrame:
    """
    Kullanıcının cilt tipi ve şikayeti girildiğinde en uygun 5 ürünü önerir.

    Algoritma:
      1. Kullanıcı girdisini TF-IDF ile vektörleştir
      2. Ürün yorumlarıyla cosine similarity hesapla
      3. Sentiment skoru ve positive_ratio ile ağırlıklandır
      4. Skor = 0.5*cosine + 0.3*avg_sentiment + 0.2*positive_ratio

    Args:
        user_input         : Kullanıcının serbest metin girişi
        top_n              : Kaç ürün önerilsin
        min_positive_ratio : Minimum pozitif yorum oranı filtresi
        category_filter    : Belirli bir kategoriyle sınırla (opsiyonel)

    Returns:
        pd.DataFrame: Top-N önerilen ürünler ve açıklama skorları
    """
    profile = extract_skin_profile(user_input)

    # Arama sorgusunu zenginleştir
    enriched_query = (
        profile['clean_input'] + ' ' +
        ' '.join(profile['detected_skin_types']) + ' ' +
        ' '.join(profile['detected_concerns'])
    )

    query_vec = product_tfidf.transform([enriched_query])
    similarities = cosine_similarity(query_vec, product_vectors).flatten()

    results = product_profiles.copy()
    results['cosine_sim'] = similarities

    # Filtrele
    results = results[results['positive_ratio'] >= min_positive_ratio]
    if category_filter:
        results = results[results['primary_category'].str.lower() == category_filter.lower()]

    # Bileşik skor
    results['final_score'] = (
        0.50 * results['cosine_sim'] +
        0.30 * results['avg_sentiment_score'] +
        0.20 * results['positive_ratio']
    )

    top = results.nlargest(top_n, 'final_score')[[
        'product_name', 'brand_name', 'primary_category',
        'price_usd', 'avg_rating', 'review_count',
        'avg_sentiment_score', 'positive_ratio', 'cosine_sim', 'final_score'
    ]].reset_index(drop=True)

    top.index = top.index + 1  # 1'den başlasın
    return top, profile


print('✅ Öneri motoru hazır!')

## 11. Öneri Demo — Örnek Kullanım

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Örnek 1: Yağlı cilt + akne
# ══════════════════════════════════════════════════════════════════════════════

user_query_1 = """
I have oily and acne prone skin. My pores are very visible and I get breakouts
frequently. I'm looking for something that helps with blemishes and minimizes pores.
"""

recs_1, profile_1 = recommend_products(user_query_1, top_n=5)

print('🔍 Tespit Edilen Profil:')
print(f'   Cilt Tipi  : {profile_1["detected_skin_types"]}')
print(f'   Şikayetler : {profile_1["detected_concerns"]}')
print()
print('💡 Önerilen Ürünler (Top-5):')
display(recs_1.style.format({
    'price_usd': '${:.2f}',
    'avg_rating': '{:.2f}',
    'avg_sentiment_score': '{:.3f}',
    'positive_ratio': '{:.1%}',
    'cosine_sim': '{:.4f}',
    'final_score': '{:.4f}',
}).background_gradient(subset=['final_score'], cmap='Greens'))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Örnek 2: Kuru cilt + yaşlanma karşıtı
# ══════════════════════════════════════════════════════════════════════════════

user_query_2 = """
My skin is very dry and dehydrated. I have fine lines and want something
with anti-aging and brightening effects. Hyaluronic acid or retinol would be great.
"""

recs_2, profile_2 = recommend_products(user_query_2, top_n=5)

print('🔍 Tespit Edilen Profil:')
print(f'   Cilt Tipi  : {profile_2["detected_skin_types"]}')
print(f'   Şikayetler : {profile_2["detected_concerns"]}')
print()
print('💡 Önerilen Ürünler (Top-5):')
display(recs_2.style.format({
    'price_usd': '${:.2f}',
    'avg_rating': '{:.2f}',
    'avg_sentiment_score': '{:.3f}',
    'positive_ratio': '{:.1%}',
    'cosine_sim': '{:.4f}',
    'final_score': '{:.4f}',
}).background_gradient(subset=['final_score'], cmap='Greens'))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Örnek 3: Hassas cilt + kızarıklık — kategori filtreli
# ══════════════════════════════════════════════════════════════════════════════

user_query_3 = """
I have very sensitive skin with redness and irritation. Looking for gentle
soothing products that calm inflammation and are fragrance-free.
"""

recs_3, profile_3 = recommend_products(
    user_query_3, top_n=5,
    category_filter=None   # ← belirli bir kategori için 'Moisturizers' gibi yazabilirsiniz
)

print('🔍 Tespit Edilen Profil:')
print(f'   Cilt Tipi  : {profile_3["detected_skin_types"]}')
print(f'   Şikayetler : {profile_3["detected_concerns"]}')
print()
print('💡 Önerilen Ürünler (Top-5):')
display(recs_3.style.format({
    'price_usd': '${:.2f}',
    'avg_rating': '{:.2f}',
    'avg_sentiment_score': '{:.3f}',
    'positive_ratio': '{:.1%}',
    'cosine_sim': '{:.4f}',
    'final_score': '{:.4f}',
}).background_gradient(subset=['final_score'], cmap='Greens'))

## 12. Öneri Motoru — Görselleştirme

In [ ]:
# ── Öneri skorları bar grafiği ────────────────────────────────────────────────

def plot_recommendations(recs: pd.DataFrame, title: str = 'Top-5 Önerilen Ürünler'):
    fig, ax = plt.subplots(figsize=(11, 5))
    labels = recs['product_name'].str[:40] + ' (' + recs['brand_name'] + ')'
    colors = plt.cm.RdYlGn(np.linspace(0.4, 0.9, len(recs))[::-1])

    bars = ax.barh(labels[::-1], recs['final_score'][::-1], color=colors, edgecolor='white', height=0.6)

    for bar, score in zip(bars, recs['final_score'][::-1]):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f'{score:.3f}', va='center', fontsize=9)

    ax.set_xlabel('Final Score')
    ax.set_title(title, fontsize=13)
    ax.set_xlim(0, recs['final_score'].max() * 1.15)
    plt.tight_layout()
    return fig

fig1 = plot_recommendations(recs_1, 'Öneriler: Yağlı + Akne Eğilimli Cilt')
fig1.savefig(METRICS_DIR / 'recommendations_oily_acne.png', bbox_inches='tight')
plt.show()

fig2 = plot_recommendations(recs_2, 'Öneriler: Kuru Cilt + Anti-Aging')
fig2.savefig(METRICS_DIR / 'recommendations_dry_antiaging.png', bbox_inches='tight')
plt.show()

## 13. Öneri Modelini Kaydet

In [ ]:
# ── Öneri motoru bileşenlerini kaydet ────────────────────────────────────────

recommendation_artifacts = {
    'product_profiles':  product_profiles,
    'product_tfidf':     product_tfidf,
    'product_vectors':   product_vectors,
    'sentiment_pipeline': best_pipeline,
    'best_model_name':   best_model_name,
}

rec_path = MODELS_DIR / 'recommendation_engine.pkl'
with open(rec_path, 'wb') as f:
    pickle.dump(recommendation_artifacts, f)

print(f'✅ Öneri motoru kaydedildi → {rec_path}')
print(f'   Ürün profili sayısı     : {len(product_profiles)}')
print(f'   Sentiment modeli        : {best_model_name}')

## 14. Çıktı Özeti

In [ ]:
print('=' * 60)
print('📁 KAYDEDILEN DOSYALAR')
print('=' * 60)

print('\n📂 outputs/models/')
for f in sorted(MODELS_DIR.glob('*')):
    size_kb = f.stat().st_size / 1024
    print(f'   {f.name:<45} {size_kb:>7.1f} KB')

print('\n📂 outputs/metrics/')
for f in sorted(METRICS_DIR.glob('*')):
    size_kb = f.stat().st_size / 1024
    print(f'   {f.name:<45} {size_kb:>7.1f} KB')

print('\n' + '=' * 60)
print('🏆 MODEL KARŞILAŞTIRMA ÖZETI')
print('=' * 60)
display(comparison_df[['test_f1', 'test_accuracy', 'test_auc', 'train_time_sec']]
        .sort_values('test_f1', ascending=False))

print(f'\n🥇 En İyi Model : {best_model_name}')
print(f'   Test F1      : {all_results[best_model_name]["metrics"]["test_f1"]}')
print(f'   Test Accuracy: {all_results[best_model_name]["metrics"]["test_accuracy"]}')